# Lecture 13: Fine-Tuning Transformers
### NLP Course 2027

---

## Learning Outcomes
- Understand fine-tuning vs feature extraction
- Fine-tune DistilBERT for text classification
- Fine-tune for token classification (NER)
- Use the Hugging Face `Trainer` API effectively

**Primary Reference:** *NLP with Transformers* Ch.2 (text classification) & Ch.4 (NER)

## 1. Transfer Learning for NLP

```
Pretrain on huge unlabeled corpus
(BERT: 16GB Wikipedia + BookCorpus)
              ↓
  General language understanding:
  grammar, facts, semantics, context
              ↓
  Fine-tune on small labeled dataset
  (e.g., 1000 sentiment-labeled tweets)
              ↓
  Task-specific model: sentiment classifier
```

### Two Strategies
| Strategy | What you do | When to use |
|----------|-------------|-------------|
| **Feature extraction** | Freeze BERT, only train new head | Very small datasets, fast |
| **Fine-tuning** | Update all weights (BERT + head) | Best results, medium+ data |
| **Full pretraining** | Train from scratch | Domain-specific (BioBERT) |

In [18]:
# !pip install transformers datasets evaluate accelerate

from datasets import load_dataset
from transformers import AutoTokenizer

# Load SST-2 (Stanford Sentiment Treebank)
dataset = load_dataset('sst2')
print(dataset)
print()
print('Training sample:')
print(dataset['train'][0])
print()
print('Label map: 0=negative, 1=positive')

DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label'],
        num_rows: 1821
    })
})

Training sample:
{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0}

Label map: 0=negative, 1=positive


In [19]:
# Tokenize the dataset
MODEL_NAME = 'distilbert-base-uncased'
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def tokenize_function(examples):
    return tokenizer(
        examples['sentence'],
        padding='max_length',
        truncation=True,
        max_length=128
    )

tokenized_datasets = dataset.map(tokenize_function, batched=True)
print('Tokenized dataset:')
print(tokenized_datasets)
print()
print('Sample tokenized:')
print(tokenized_datasets['train'][0])

Tokenized dataset:
DatasetDict({
    train: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 67349
    })
    validation: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 872
    })
    test: Dataset({
        features: ['idx', 'sentence', 'label', 'input_ids', 'attention_mask'],
        num_rows: 1821
    })
})

Sample tokenized:
{'idx': 0, 'sentence': 'hide new secretions from the parental units ', 'label': 0, 'input_ids': [101, 5342, 2047, 3595, 8496, 2013, 1996, 18643, 3197, 102, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0], 'attention_mask': [1, 1, 1, 1, 1, 1, 1, 

In [20]:
# Set up the model for classification
from transformers import AutoModelForSequenceClassification
import torch

model = AutoModelForSequenceClassification.from_pretrained(
    MODEL_NAME,
    num_labels=2  # binary: positive/negative
)
print('Model architecture (classification head added):')
print(model.classifier)  # The new head
print(f'\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}')
print(f'Trainable parameters: {sum(p.numel() for p in model.parameters() if p.requires_grad):,}')

Some weights of DistilBertForSequenceClassification were not initialized from the model checkpoint at distilbert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight', 'pre_classifier.bias', 'pre_classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


Model architecture (classification head added):
Linear(in_features=768, out_features=2, bias=True)

Total parameters: 66,955,010
Trainable parameters: 66,955,010


In [21]:
from transformers import TrainingArguments, Trainer
import evaluate
import numpy as np

# Load accuracy metric
metric = evaluate.load('accuracy')

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    predictions = np.argmax(logits, axis=-1)
    return metric.compute(predictions=predictions, references=labels)

# Training configuration
training_args = TrainingArguments(
    output_dir='./results/sentiment',
    num_train_epochs=3,
    per_device_train_batch_size=32,
    per_device_eval_batch_size=64,
    warmup_steps=500,
    weight_decay=0.01,
    logging_dir='./logs',
    logging_steps=100,
    evaluation_strategy='epoch',
    save_strategy='epoch',
    load_best_model_at_end=True,
    metric_for_best_model='accuracy',
    push_to_hub=False,
)
print('Training arguments set!')
print(f'Epochs: {training_args.num_train_epochs}')
print(f'Batch size: {training_args.per_device_train_batch_size}')

Training arguments set!
Epochs: 3
Batch size: 32


In [22]:
# Create Trainer and train
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets['train'],
    eval_dataset=tokenized_datasets['validation'],
    tokenizer=tokenizer,
    data_collator=data_collator,
    compute_metrics=compute_metrics,
)

# NOTE: Actually running training takes 10-30 min on CPU, ~2 min on GPU
# Uncomment to train:
# trainer.train()
# results = trainer.evaluate()
# print(results)

print('Trainer configured. Call trainer.train() to start fine-tuning.')
print('Expected accuracy on SST-2 validation set: ~91-92%')

Trainer configured. Call trainer.train() to start fine-tuning.
Expected accuracy on SST-2 validation set: ~91-92%


In [23]:
# Inference with fine-tuned model
from transformers import pipeline

# Use a pre-fine-tuned model from Hub (avoid training locally)
sentiment_model = pipeline(
    'text-classification',
    model='distilbert-base-uncased-finetuned-sst-2-english'
)

test_sentences = [
    'This transformer course is absolutely brilliant!',
    'Fine-tuning BERT was incredibly confusing and frustrating.',
    'The model performance was acceptable but not outstanding.',
    'I cannot believe how easy Hugging Face makes everything.',
]
for text in test_sentences:
    result = sentiment_model(text)[0]
    print(f'{result["label"]:10s} ({result["score"]:.3f})  {text}')

Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


POSITIVE   (1.000)  This transformer course is absolutely brilliant!
NEGATIVE   (0.999)  Fine-tuning BERT was incredibly confusing and frustrating.
NEGATIVE   (0.994)  The model performance was acceptable but not outstanding.
POSITIVE   (0.996)  I cannot believe how easy Hugging Face makes everything.


## 2. Fine-Tuning for Token Classification (NER)

NER is a **token classification** task: each token gets a label.

```
Sentence:  Apple  was  founded  by  Steve  Jobs
Tokens:    Ap ##ple was  founded  by  Steve  Jobs
Labels:    B-ORG I-ORG O    O      O   B-PER  I-PER
```

### Challenge: Subword Tokenization + Labels
BERT tokenizes 'Apple' → ['Ap', '##ple'] but we only have one label.
- Common approach: label the **first subword**, assign `-100` to others
- `-100` tells PyTorch to **ignore** those positions in loss computation

### CoNLL-2003 IOB Format
```
B-PER = Beginning of a Person entity
I-PER = Inside a Person entity
B-ORG = Beginning of an Organization
O     = Outside any entity
```

In [24]:
# Load CoNLL-2003 NER dataset
ner_dataset = load_dataset('conll2003')
print(ner_dataset)
print()
print('Sample:')
sample = ner_dataset['train'][0]
print('Tokens:', sample['tokens'])
print('NER tags:', sample['ner_tags'])

label_names = ner_dataset['train'].features['ner_tags'].feature.names
print('Label names:', label_names)

Using the latest cached version of the dataset since conll2003 couldn't be found on the Hugging Face Hub
Found the latest cached dataset configuration 'conll2003' at /Users/lei/.cache/huggingface/datasets/conll2003/conll2003/1.0.0/9a4d16a94f8674ba3466315300359b0acd891b68b6c8743ddf60b9c702adce98 (last modified on Mon Mar 30 15:13:27 2026).


DatasetDict({
    train: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 14041
    })
    validation: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3250
    })
    test: Dataset({
        features: ['id', 'tokens', 'pos_tags', 'chunk_tags', 'ner_tags'],
        num_rows: 3453
    })
})

Sample:
Tokens: ['EU', 'rejects', 'German', 'call', 'to', 'boycott', 'British', 'lamb', '.']
NER tags: [3, 0, 7, 0, 0, 0, 7, 0, 0]
Label names: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [25]:
# Tokenize for NER: align labels with subword tokens
tokenizer = AutoTokenizer.from_pretrained('bert-base-cased')  # cased for NER

def tokenize_and_align_labels(examples):
    tokenized_inputs = tokenizer(
        examples['tokens'],
        truncation=True,
        is_split_into_words=True  # input is already word-tokenized
    )
    labels = []
    for i, label in enumerate(examples['ner_tags']):
        word_ids = tokenized_inputs.word_ids(batch_index=i)
        previous_word_idx = None
        label_ids = []
        for word_idx in word_ids:
            if word_idx is None:                        # special tokens
                label_ids.append(-100)
            elif word_idx != previous_word_idx:         # first subword of word
                label_ids.append(label[word_idx])
            else:                                        # subsequent subwords
                label_ids.append(-100)
            previous_word_idx = word_idx
        labels.append(label_ids)
    tokenized_inputs['labels'] = labels
    return tokenized_inputs

tokenized_ner = ner_dataset.map(tokenize_and_align_labels, batched=True)
print('NER dataset tokenized!')
print('Sample labels:', tokenized_ner['train'][0]['labels'][:20])

NER dataset tokenized!
Sample labels: [-100, 3, 0, 7, 0, 0, 0, 7, 0, -100, 0, -100]


In [26]:
# Set up NER model
from transformers import AutoModelForTokenClassification

id2label = {i: name for i, name in enumerate(label_names)}
label2id = {name: i for i, name in enumerate(label_names)}

ner_model = AutoModelForTokenClassification.from_pretrained(
    'bert-base-cased',
    num_labels=len(label_names),
    id2label=id2label,
    label2id=label2id
)
print(f'NER model ready. Labels: {label_names}')

Some weights of BertForTokenClassification were not initialized from the model checkpoint at bert-base-cased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


NER model ready. Labels: ['O', 'B-PER', 'I-PER', 'B-ORG', 'I-ORG', 'B-LOC', 'I-LOC', 'B-MISC', 'I-MISC']


In [27]:
# Evaluate NER with seqeval
# !pip install seqeval
import evaluate as hf_eval
import seqeval

seqeval_metric = hf_eval.load('seqeval')

def compute_ner_metrics(p):
    predictions, labels = p
    predictions = np.argmax(predictions, axis=2)
    true_preds = [
        [label_names[pred] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]
    true_labels = [
        [label_names[lbl] for pred, lbl in zip(prediction, label) if lbl != -100]
        for prediction, label in zip(predictions, labels)
    ]
    results = seqeval_metric.compute(predictions=true_preds, references=true_labels)
    return {
        'precision': results['overall_precision'],
        'recall': results['overall_recall'],
        'f1': results['overall_f1'],
        'accuracy': results['overall_accuracy'],
    }

print('NER metrics configured with seqeval.')
print('Seqeval evaluates at entity level (not token level).')
print('Expected F1 on CoNLL-2003 test set with BERT: ~91%')

NER metrics configured with seqeval.
Seqeval evaluates at entity level (not token level).
Expected F1 on CoNLL-2003 test set with BERT: ~91%


In [28]:
# Quick NER inference with pre-fine-tuned model
from transformers import pipeline

ner_pipeline = pipeline(
    'ner',
    model='dbmdz/bert-large-cased-finetuned-conll03-english',
    aggregation_strategy='simple'
)

text = """Tim Cook, CEO of Apple Inc., announced a new partnership with
Microsoft at the WWDC conference held in San Jose, California.
"""

print(f'Text: {text}')
entities = ner_pipeline(text)
print('Entities:')
for ent in entities:
    print(f'  [{ent["entity_group"]:6s}] {ent["word"]:25s}  score={ent["score"]:.3f}')

Some weights of the model checkpoint at dbmdz/bert-large-cased-finetuned-conll03-english were not used when initializing BertForTokenClassification: ['bert.pooler.dense.bias', 'bert.pooler.dense.weight']
- This IS expected if you are initializing BertForTokenClassification from the checkpoint of a model trained on another task or with another architecture (e.g. initializing a BertForSequenceClassification model from a BertForPreTraining model).
- This IS NOT expected if you are initializing BertForTokenClassification from the checkpoint of a model that you expect to be exactly identical (initializing a BertForSequenceClassification model from a BertForSequenceClassification model).
Hardware accelerator e.g. GPU is available in the environment, but no `device` argument is passed to the `Pipeline` object. Model will be on CPU.


Text: Tim Cook, CEO of Apple Inc., announced a new partnership with
Microsoft at the WWDC conference held in San Jose, California.

Entities:
  [PER   ] Tim Cook                   score=1.000
  [ORG   ] Apple Inc                  score=0.999
  [ORG   ] Microsoft                  score=1.000
  [MISC  ] WWDC                       score=0.982
  [LOC   ] San Jose                   score=0.998
  [LOC   ] California                 score=0.999


## 3. Training Tips for Fine-Tuning

| Hyperparameter | Typical Range | Notes |
|----------------|--------------|-------|
| Learning rate | 2e-5 to 5e-5 | Lower than regular DL |
| Epochs | 2–5 | More → overfitting |
| Batch size | 16–32 | Larger = more stable |
| Warmup steps | 10% of training | Prevents early instability |
| Weight decay | 0.01 | L2 regularization |
| Max length | 128–512 | Tradeoff: context vs speed |

## Practice Exercises

See **`Lecture-13-Homework.ipynb`** for the practice exercises accompanying this lecture.

## Summary

| Task | Model Type | Dataset Format |
|------|------------|----------------|
| Classification | `AutoModelForSequenceClassification` | (text, label) |
| NER | `AutoModelForTokenClassification` | (tokens, ner_tags) |
| QA | `AutoModelForQuestionAnswering` | (question, context, answer) |

**Fine-tuning recipe:**
1. Load tokenizer + model
2. Tokenize dataset with `.map()`
3. Configure `TrainingArguments`
4. Create `Trainer` and call `.train()`

**Next Lecture**: Advanced Transformer Topics — cross-lingual models, efficiency methods, LoRA.

---
*Book reference: NLP with Transformers Ch.2 (classification), Ch.4 (NER)*

---
**Author: Lei Wu | © 2026 Lei Wu. All rights reserved. Unauthorized use is prohibited.**